In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df_original = pd.read_csv('../polygraphpy/data/polarizability_data.csv')
df_ga_output = pd.read_csv('../polygraphpy/data/ga_output/generated_molecules.csv')
df_gpt_output = pd.read_csv('../polygraphpy/data/generative_output/generated_molecules.csv')

df_gpt_output = df_gpt_output.rename(columns={'smiles_A': 'smiles'})
df_original = df_original[df_original['chain_size'] == 0]

scaler = MinMaxScaler()
scaler.fit(df_original[['static_polarizability']])

df_ga_output['static_polarizability_pred_original'] = scaler.inverse_transform(df_ga_output[['static_polarizability']])

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors

def get_precise_vdw_volume(smiles):
    if pd.isna(smiles):
        return None
        
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
        
    mol = Chem.AddHs(mol)
    
    params = AllChem.ETKDG()
    params.randomSeed = 42 
    success = AllChem.EmbedMolecule(mol, params)
    
    if success != 0:
        pass
        
    try:
        AllChem.UFFOptimizeMolecule(mol)
    except:
        pass
        
    try:
        dclv = rdMolDescriptors.DoubleCubicLatticeVolume(mol)
        return dclv.GetVDWVolume()
    except:
        pass

df_ga_output['vdw_volume'] = df_ga_output['smiles'].apply(get_precise_vdw_volume)
df_gpt_output['vdw_volume'] = df_gpt_output['smiles'].apply(get_precise_vdw_volume)

[13:03:24] Conflicting single bond directions around double bond at index 5.
[13:03:24]   BondStereo set to STEREONONE and single bond directions set to NONE.
[13:03:25] Conflicting single bond directions around double bond at index 5.
[13:03:25]   BondStereo set to STEREONONE and single bond directions set to NONE.
[13:03:25] Conflicting single bond directions around double bond at index 5.
[13:03:25]   BondStereo set to STEREONONE and single bond directions set to NONE.
[13:03:29] Conflicting single bond directions around double bond at index 19.
[13:03:29]   BondStereo set to STEREONONE and single bond directions set to NONE.
[13:03:41] Conflicting single bond directions around double bond at index 11.
[13:03:41]   BondStereo set to STEREONONE and single bond directions set to NONE.
[13:03:41] Conflicting single bond directions around double bond at index 8.
[13:03:41]   BondStereo set to STEREONONE and single bond directions set to NONE.
[13:03:42] Conflicting single bond direction

In [ ]:
best_indices = df_ga_output.groupby('static_polarizability')['fitness'].idxmin()
df_ga_output = df_ga_output.loc[best_indices]
df_ga_output = df_ga_output.reset_index(drop=True)

df_gpt_output = df_gpt_output.sort_values(by='error', ascending=True).head(100)

In [16]:
import numpy as np

def calculate_refractive_index(df, pol_col='static_polarizability_pred_original', vol_col='vdw_volume'):
    """
    Computes Lorentz-Lorentz refractive index.
    
    Formula:
      n_o^2 = [1 + 2 * A] / [1 - A]
      where A = (4*pi/3) * N * alpha
            alpha = static_polarizability * 0.14818471
            N = K / vdw_volume
            K = 0.68
    """
    K = 0.68
    PI = np.pi
    
    N = K / df[vol_col]
    
    alpha = df[pol_col] * 0.14818471
    
    A = (4 * PI / 3) * N * alpha
    
    n_squared = (1 + 2 * A) / (1 - A)
    
    n_o = np.sqrt(n_squared)
    
    return n_o

df_ga_output['refractive_index'] = calculate_refractive_index(df_ga_output, pol_col='static_polarizability_pred_original', vol_col='vdw_volume')

df_gpt_output['refractive_index'] = calculate_refractive_index(df_gpt_output, pol_col='static_polarizability_pred_original', vol_col='vdw_volume')

/home/jgduarte/Documents/RA/Projects/3M/PolyGraphPy/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [17]:
df_ga_output

,smiles,static_polarizability,fitness,static_polarizability_pred_original,vdw_volume,refractive_index
0,COC(=O)C=C(CCl)C(=O)OC,0.127795,-0.126169,127.535665,162.080036,1.578565
1,COC(=O)C=CC(=O)OC,0.128977,-0.127881,128.457782,128.589674,1.785272
2,COC(=O)C=C(C(=O)OC)P(=O)(OC)OC,0.130158,-0.121350,129.379899,202.513662,1.451780
3,COC(=O)C=CC(=O)OC,0.131340,-0.121717,130.302016,128.589674,1.800574
4,COC(=O)C(CN)=C(CBr)C(=O)OC,0.132522,-0.125795,131.224133,174.335207,1.548209
...,...,...,...,...,...,...
95,COC(=O)C(C#N)=C(C#N)C(=O)OC,0.240064,-0.237415,215.136781,166.634011,2.143034
96,COC(=O)C(C#N)=C(C#N)C(=O)OC,0.241246,-0.225693,216.058898,166.634011,2.150956
97,CC(C(=O)NC=C(C#N)C(=O)Oc1ccc([N+](=O)[O-])cc1)...,0.242428,-0.130207,216.981015,399.134934,1.375995
98,COC(=O)C(C#N)=C(C#N)C(=O)OC,0.243610,-0.238304,217.903132,166.634011,2.166956


In [18]:
df_gpt_output

,smiles,static_polarizability,static_polarizability_pred,static_polarizability_original,static_polarizability_pred_original,error,vdw_volume,refractive_index
114,CCOC(=O)/C=C/C1=C(N(N=C1)C2=CC=CC=C2CC3=C)C=CC=C3,0.211055,0.210951,192.501723,192.420680,0.000421,322.603906,1.417532
82,CCCCCCCCCCCCCCCCOC(=O)C=C,0.160804,0.161178,153.291951,153.583530,0.001902,320.607942,1.326770
142,CC(C)COC1=CC=CC=C1OC(=O)/C=C/C2=C(C(=CC=C2)OC)...,0.256281,0.257868,227.790518,229.028120,0.005433,385.210311,1.416011
35,C=CC(=O)OCC1=C(C=CC=N1)[N+1](=O)[O-1],0.075377,0.076139,86.635338,87.229930,0.006863,176.920373,1.337309
330,CC(=O)NC=CC1=C(C=C(C=C1)C#N)OC[C@@H1](CCC(=O)O...,0.502513,0.498223,419.918403,416.571700,0.007970,514.182280,1.599682
...,...,...,...,...,...,...,...,...
104,CCOC(=O)/C=C/C1=C(N(N=C1C)C)N2C=CC3=C2N=CC=C3,0.201005,0.159665,184.659769,152.403470,0.174680,289.796360,1.362319
165,CC(=O)NC(=CC1=C(C=C(C=C1)C#N)OC[C@@H1](CCC(=O)...,0.296482,0.354637,259.158336,304.535250,0.175093,472.410647,1.456506
150,COC(=O)/C(=C/C1=CC=C(O1)C2=CC=C(C=C2)S(=O)(=O)...,0.271357,0.217514,239.553450,197.541000,0.175378,345.544686,1.397899
80,CCCCCCCCCCCCCOC(=O)C=C,0.160804,0.126224,153.291951,126.310074,0.176016,273.922254,1.313392
